# Fixed best submission pipeline

This notebook uses the best validated single model from the local experiments:

- Model: LightGBM
- 5-fold OOF AUC: 0.958734
- Fixed parameters, no GridSearch and no model comparison
- Run all cells to generate `data/submission.csv`

The notebook trains once on all training rows and writes the final prediction directly to the `data` folder.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
if not (PROJECT_DIR / "data" / "train.csv").exists():
    PROJECT_DIR = Path.home() / "Predicting Airline Satisfaction"
DATA_DIR = PROJECT_DIR / "data"

MODEL_PARAMS = {
    "objective": "binary",
    "metric": "auc",
    "n_estimators": 600,
    "learning_rate": 0.03,
    "num_leaves": 63,
    "min_child_samples": 20,
    "subsample": 0.85,
    "subsample_freq": 1,
    "colsample_bytree": 0.7,
    "reg_lambda": 2.0,
    "n_jobs": 8,
    "random_state": 42,
    "verbosity": -1,
}

print("Project:", PROJECT_DIR)
print("Model: LightGBM")
print(MODEL_PARAMS)

Project: C:\Users\User\Desktop\Predicting Airline Satisfaction
Model: LightGBM
{'objective': 'binary', 'metric': 'auc', 'n_estimators': 600, 'learning_rate': 0.03, 'num_leaves': 63, 'min_child_samples': 20, 'subsample': 0.85, 'subsample_freq': 1, 'colsample_bytree': 0.7, 'reg_lambda': 2.0, 'n_jobs': 8, 'random_state': 42, 'verbosity': -1}


In [2]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

arrival_median = train["Arrival Delay in Minutes"].median()
train["Arrival Delay in Minutes"] = train["Arrival Delay in Minutes"].fillna(arrival_median)
test["Arrival Delay in Minutes"] = test["Arrival Delay in Minutes"].fillna(arrival_median)

X = train.drop(columns=["satisfaction", "id"])
y = train["satisfaction"].astype(int)
test_ids = test["id"].copy()
X_test = test.drop(columns=["id"])

categorical_cols = ["Gender", "Customer Type", "Type of Travel", "Class"]
X = pd.get_dummies(X, columns=categorical_cols, dtype=np.int8)
X_test = pd.get_dummies(X_test, columns=categorical_cols, dtype=np.int8)
X_test = X_test.reindex(columns=X.columns, fill_value=0)

print("Train:", X.shape)
print("Test:", X_test.shape)
print("Positive rate:", float(y.mean()))

Train: (699635, 26)
Test: (299844, 26)
Positive rate: 0.44357272006117476


In [3]:
model = LGBMClassifier(**MODEL_PARAMS)
model.fit(X, y)

test_pred = model.predict_proba(X_test)[:, 1]
submission = pd.DataFrame(
    {
        "id": test_ids.to_numpy(),
        "satisfaction": np.clip(test_pred, 0.0, 1.0),
    }
)

assert submission.columns.tolist() == sample_submission.columns.tolist()
assert len(submission) == len(sample_submission)
assert np.array_equal(submission["id"].to_numpy(), sample_submission["id"].to_numpy())
assert submission["satisfaction"].between(0.0, 1.0).all()

submission_path = DATA_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)

print("Saved:", submission_path)
print(submission.head())

Saved: C:\Users\User\Desktop\Predicting Airline Satisfaction\data\submission.csv
       id  satisfaction
0  699635      0.224981
1  699636      0.890111
2  699637      0.916821
3  699638      0.070004
4  699639      0.050656
